# Notebook 01 — Exploratory Data Analysis

**Goal:** Understand the structure, quality, and distributions of the Starbucks dataset before any causal analysis.

## Dataset Overview

| File | Rows | Description |
|---|---|---|
| `portfolio.json` | 10 | Offer metadata: type, difficulty, reward, duration, channels |
| `profile.json` | 17,000 | Customer demographics: age, gender, income, membership date |
| `transcript.json` | ~306,000 | Events: offer received, viewed, completed, transaction |

## Key Questions
1. What types of offers exist and how are they structured?
2. Who are our customers? (demographics)
3. What does the offer funnel look like? (received → viewed → completed)

In [1]:
import sys
from pathlib import Path
_root = Path().resolve(); _root = _root.parent if _root.name == 'notebooks' else _root; sys.path.insert(0, str(_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.extract import extract
from src.transform import clean_profile, expand_transcript_value
from src.constants import REPORTS_FIGURES
from src.utils.plot import save_fig, funnel_chart

REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette='muted')

## 1. Load Raw Data

In [2]:
portfolio, profile, transcript = extract()
print(f'portfolio : {portfolio.shape}')
print(f'profile   : {profile.shape}')
print(f'transcript: {transcript.shape}')

portfolio : (10, 6)
profile   : (17000, 5)
transcript: (306534, 4)


## 2. Portfolio — Offer Catalog

There are **3 offer types**:
- **BOGO** (Buy One Get One): spend X to get a reward of X
- **Discount**: spend X to get a reward of Y (Y < X)
- **Informational**: no reward, purely a notification (no completion event exists)

In [3]:
portfolio[['id','offer_type','difficulty','reward','duration','channels']]

,id,offer_type,difficulty,reward,duration,channels
0,ae264e3637204a6fb9bb56bc8210ddfd,bogo,10,10,7,"[email, mobile, social]"
1,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,10,10,5,"[web, email, mobile, social]"
2,3f207df678b143eea3cee63160fa8bed,informational,0,0,4,"[web, email, mobile]"
3,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,5,5,7,"[web, email, mobile]"
4,0b1e1539f2cc45b7b9fa7c272da2e1d7,discount,20,5,10,"[web, email]"
5,2298d6c36e964ae4a3e7e9706d1fb8c2,discount,7,3,7,"[web, email, mobile, social]"
6,fafdcd668e3743c1bb461111dcafc2a4,discount,10,2,10,"[web, email, mobile, social]"
7,5a8bc65990b245e5a138643cd4eb9837,informational,0,0,3,"[email, mobile, social]"
8,f19421c1d4aa40978ebb69ca19b0e20d,bogo,5,5,5,"[web, email, mobile, social]"
9,2906b810c7d4411798c6938adc9daaa5,discount,10,2,7,"[web, email, mobile]"


In [4]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

portfolio['offer_type'].value_counts().plot(kind='bar', ax=axes[0], color='#00704A', title='Offers by Type')
axes[0].set_xlabel('')

portfolio.groupby('offer_type')['difficulty'].mean().plot(kind='bar', ax=axes[1], color='#CBA258', title='Avg Difficulty by Type')
axes[1].set_ylabel('Spend Required ($)')
axes[1].set_xlabel('')

portfolio.groupby('offer_type')['reward'].mean().plot(kind='bar', ax=axes[2], color='#1a1a2e', title='Avg Reward by Type')
axes[2].set_ylabel('Reward ($)')
axes[2].set_xlabel('')

for ax in axes:
    ax.tick_params(axis='x', rotation=0)

fig.tight_layout()
save_fig(fig, REPORTS_FIGURES / '01_portfolio_overview.png')
plt.show()

## 3. Profile — Customer Demographics

**Important data quality note:** Customers who did not provide demographic data have `age = 118` as a sentinel value and `income = NaN`. These rows are removed during cleaning.

In [5]:
print(f'Missing demographics (age=118): {(profile["age"] == 118).sum():,} customers')
print(f'Missing income: {profile["income"].isna().sum():,} customers')
profile_clean = clean_profile(profile)
print(f'Valid customers after cleaning: {len(profile_clean):,}')

Missing demographics (age=118): 2,175 customers
Missing income: 2,175 customers
Valid customers after cleaning: 14,825


In [6]:
profile_clean[['age', 'income', 'membership_days']].describe().round(1)

,age,income,membership_days
count,14825.0,14825.0,14825.0
mean,54.4,65405.0,528.5
std,17.4,21598.3,419.2
min,18.0,30000.0,6.0
25%,42.0,49000.0,214.0
50%,55.0,64000.0,364.0
75%,66.0,80000.0,803.0
max,101.0,120000.0,1829.0


In [7]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

profile_clean['age'].plot(kind='hist', bins=30, ax=axes[0,0], color='#00704A', title='Age Distribution')
axes[0,0].set_xlabel('Age')

profile_clean['income'].plot(kind='hist', bins=30, ax=axes[0,1], color='#CBA258', title='Income Distribution')
axes[0,1].set_xlabel('Annual Income ($)')

profile_clean['gender'].value_counts().plot(kind='bar', ax=axes[1,0], color=['#00704A','#CBA258','#C0C0C0'], title='Gender')
axes[1,0].set_xlabel('')
axes[1,0].tick_params(axis='x', rotation=0)

profile_clean['age_group'].value_counts().sort_index().plot(kind='bar', ax=axes[1,1], color='#1a1a2e', title='Age Groups')
axes[1,1].set_xlabel('')
axes[1,1].tick_params(axis='x', rotation=0)

fig.suptitle('Customer Demographics Overview', fontsize=14, y=1.01)
fig.tight_layout()
save_fig(fig, REPORTS_FIGURES / '01_demographics.png')
plt.show()

## 4. Transcript — Event Distribution & Offer Funnel

The transcript records 4 event types:
- `offer received`: customer was sent an offer
- `offer viewed`: customer opened/saw the offer
- `offer completed`: customer completed the offer requirements
- `transaction`: any purchase, with or without an active offer

In [8]:
transcript_exp = expand_transcript_value(transcript)
transcript_exp['event'].value_counts()

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64

In [9]:
# Overall funnel across all offers
received = (transcript_exp['event'] == 'offer received').sum()
viewed   = (transcript_exp['event'] == 'offer viewed').sum()
completed = (transcript_exp['event'] == 'offer completed').sum()

print(f'Received  : {received:>8,}')
print(f'Viewed    : {viewed:>8,}  ({viewed/received:.1%} of received)')
print(f'Completed : {completed:>8,}  ({completed/received:.1%} of received)')

Received  :   76,277
Viewed    :   57,725  (75.7% of received)
Completed :   33,579  (44.0% of received)


In [10]:
fig = funnel_chart(
    stages=['Received', 'Viewed', 'Completed'],
    counts=[received, viewed, completed],
    title='Overall Offer Funnel — All Types',
    save_path=REPORTS_FIGURES / '01_overall_funnel.png',
)
plt.show()

In [11]:
# Funnel per offer type
events_with_type = (
    transcript_exp[transcript_exp['event'].isin(['offer received','offer viewed','offer completed'])]
    .merge(portfolio[['id','offer_type']], left_on='offer_id', right_on='id', how='left')
)

funnel = (
    events_with_type.groupby(['offer_type','event'])
    .size()
    .unstack(fill_value=0)
    .rename(columns=lambda c: c.replace('offer ',''))
)
funnel['view_rate']     = (funnel['viewed']    / funnel['received'] * 100).round(1)
funnel['complete_rate'] = (funnel['completed'] / funnel['received'] * 100).round(1)
funnel

event,completed,received,viewed,view_rate,complete_rate
offer_type,,,,,
bogo,15669,30499,25449,83.4,51.4
discount,17910,30543,21445,70.2,58.6
informational,0,15235,10831,71.1,0.0


## 5. Key EDA Takeaways

Write your observations here after running the notebook. Example:
- X% of customers have missing demographics
- BOGO and Discount have similar view rates (~X%)
- Informational offers have no completion events (by design)
- The customer base skews toward ages 45-65 with mid-to-high income